# Junior AI Investigator: fraud referral triage

The fraud engine flags home-care claims; most turn out to be false alarms. For each of the 50
referrals, the AI scores the risk, explains it in plain English and suggests what to do next.
Every fact it cites is checked against the data, and hard evidence (double billing, overlapping
visits, a shared claim number) can never be closed. The investigator makes the final call.

**In Colab:** add your Groq key under Secrets (🔑, left sidebar) as `GROQ_API_KEY`, then
*Runtime → Run all*. When asked, upload `sample_cases.csv`, and `ai_assessments.json` to reuse
saved AI results instead of calling the AI.
**Locally:** `pip install -r requirements.txt`, put `GROQ_API_KEY=...` in `.env`, then
`jupyter notebook fraud_triage.ipynb` from the project folder.

In [ ]:
import json, os, re, subprocess, sys
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from html import escape
from pathlib import Path
from typing import Literal

import pandas as pd
from pydantic import BaseModel, Field

DATA = Path("data")
try:  # in Colab: the key from Secrets, the data uploaded
    from google.colab import files, userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "groq"], check=True)
    DATA.mkdir(exist_ok=True)
    if not (DATA / "sample_cases.csv").exists():
        print("Upload sample_cases.csv, and ai_assessments.json to reuse saved AI results:")
        for name, content in files.upload().items():
            (DATA / name).write_bytes(content)
except ImportError:  # locally, from the project folder, with the key in .env
    from dotenv import load_dotenv
    load_dotenv(".env")
from groq import Groq

## 1. The cases, and what no single row shows
Besides its own row, the AI gets the other cases with the same claim number, how the amount,
visits and distance compare with the usual (median) for that type of care, and any hard evidence.

In [ ]:
cases = pd.read_csv(DATA / "sample_cases.csv")
records = {c["case_id"]: c for c in cases.to_dict("records")}
USUAL = cases.groupby("care_type")[["claim_amount_usd", "weekly_visit_frequency",
                                    "member_provider_distance_miles"]].median()
HARD = {"duplicate_service_billed": "the same service was billed twice",
        "service_overlap_other_provider": "another provider billed for the same time"}


def facts(case_id):
    c, usual = records[case_id], USUAL.loc[records[case_id]["care_type"]]
    twins = [o for o, r in records.items() if r["claim_number"] == c["claim_number"] and o != case_id]
    return {
        # A field named "peer" nudges the AI into jargon, so it is renamed.
        "case": {("referral_system_amount_vs_similar_pct" if k == "amount_vs_peer_avg_pct" else k): v
                 for k, v in c.items()},
        "shares_claim_number_with": twins,
        "typical_for_this_care_type": {f"typical_{k}": float(v) for k, v in usual.items()},
        # Pre-computed multiples: the AI misreads "+115%" and does arithmetic badly.
        "compared_with_usual": {f"{k}_times_usual": round(c[k] / v, 1) for k, v in usual.items()},
        "hard_evidence": [why for col, why in HARD.items() if c[col] == 1]
                         + [f"the same claim number is also used on {', '.join(twins)}"] * bool(twins),
    }

## 2. The AI: what it must return, its instructions, and the Groq call

In [ ]:
Level = Literal["Low", "Medium", "High"]
Action = Literal["Close as false positive", "Request documentation", "Escalate to SIU"]


class Fact(BaseModel):
    field: str  # a key copied from the facts, so it can be checked
    value: str
    why: str


class Assessment(BaseModel):
    summary: str
    key_facts: list[Fact]
    risk_score: int = Field(ge=0, le=100)
    risk_level: Level
    confidence: Level
    recommended_action: Action
    next_steps: list[str]
    open_questions: list[str]


class Answer(BaseModel):
    answer: str
    citations: list[Fact]
    answerable: bool


FACT = '{"field": str, "value": str, "why": str}'
SHAPES = {  # far fewer tokens than a JSON schema; pydantic still checks every reply
    Assessment: '{"summary": str, "key_facts": [' + FACT + '], "risk_score": int 0-100, '
                '"risk_level": "Low"|"Medium"|"High", "confidence": "Low"|"Medium"|"High", '
                '"recommended_action": "Close as false positive"|"Request documentation"|'
                '"Escalate to SIU", "next_steps": [str], "open_questions": [str]}',
    Answer: '{"answer": str, "citations": [' + FACT + '], "answerable": bool}',
}
WRITING = """
Write for a colleague who has never seen this data: plain, everyday words about what happened.
Say "the patient", "the provider", "similar cases", "usually". Data key names (with underscores)
never appear in your text, only in a cited "field", copied exactly from the facts, e.g.
{"field": "weekly_visit_frequency", "value": "12", "why": "..."}. Use dollars, miles and
percentages of bills (0.44 is "44% of the bills"). To compare with the usual, quote
compared_with_usual (2.2 means "about 2.2 times the usual"); never do your own arithmetic.
referral_system_amount_vs_similar_pct is the referral system's own comparison."""
ASSESS = """You are a junior fraud investigator at a long-term-care insurer, doing first-pass triage
of referrals from an automated fraud engine; most turn out to be false positives. A senior
investigator reviews your work. Use ONLY the facts given; never invent data, history or documents.
- risk_score is your 0-100 judgement of how likely this is fraud; risk_level must match it: under
  20 Low, 20-49 Medium, 50 or more High. confidence is how clear-cut the evidence is.
- Warning signs at 0 are normal; none at all is evidence for a false positive.
- If hard_evidence is not empty, never rate the case Low or recommend "Close as false positive".
- summary: at most 3 short sentences (about 60 words). At most 3 next_steps (concrete actions)
  and 3 open_questions (what this data can't answer), one short sentence each.""" + WRITING
QA = """You are a junior fraud investigator answering a senior investigator's questions about ONE
referral. Use ONLY the facts given: the case, its cross-case facts and your earlier assessment. If
they can't answer, say so, set answerable to false and name the record that would. Answer in 2-5
sentences and cite each fact you rely on. You advise; the investigator decides.""" + WRITING
MODELS = ["openai/gpt-oss-120b", "openai/gpt-oss-20b"]  # the second takes over if the first is out


def ask_ai(system, user, schema):
    """A reply checked against `schema`. Tries each model with each key in GROQ_API_KEY
    (comma-separated, for more free quota); the SDK itself waits out per-minute rate limits."""
    error = None
    for model in MODELS:
        for key in os.environ["GROQ_API_KEY"].split(","):
            try:
                reply = Groq(api_key=key.strip(), max_retries=3).chat.completions.create(
                    model=model, temperature=0, reasoning_effort="low",
                    response_format={"type": "json_object"},
                    messages=[{"role": "system", "content": f"{system}\nReply with only JSON "
                                                            f"shaped like: {SHAPES[schema]}"},
                              {"role": "user", "content": user}])
                return schema.model_validate_json(reply.choices[0].message.content)
            except Exception as e:  # out of quota, bad key or invalid reply: try the next
                error = e
    raise RuntimeError(f"The AI is unavailable: {error}")

## 3. Assess every case, then check the AI's work
Results are saved to `data/ai_assessments.json`, so later runs (and anyone without a key) reuse
them; delete it to assess again. Every fact the AI cites is checked against the data, and a case
with hard evidence that the AI would close or rate Low is flagged.

In [ ]:
SAVED = DATA / "ai_assessments.json"
assessments = json.loads(SAVED.read_text()) if SAVED.exists() else {}


def assess(case_id):
    try:
        return case_id, ask_ai(ASSESS, f"Assess this referral.\n{json.dumps(facts(case_id))}",
                               Assessment).model_dump()
    except RuntimeError as e:
        print(case_id, e)
        return case_id, None


with ThreadPoolExecutor(4) as pool:  # a few at once; run the cell again to retry any that failed
    todo = [i for i in records if i not in assessments]
    assessments |= {i: a for i, a in pool.map(assess, todo) if a}
SAVED.write_text(json.dumps(assessments, indent=1))
print(f"{len(assessments)} of {len(records)} cases assessed")


def matches(cited, actual):
    """Does the value the AI cited match the data? Tolerates "$32,373", "205 miles", "44%" for 0.44."""
    cited = {"yes": "1", "true": "1", "no": "0", "false": "0"}.get(cited.strip().lower(), cited)
    if isinstance(actual, list):  # case ids; an empty list is a checkable "none"
        return all(str(x) in cited for x in actual) if actual else cited.strip().lower() in ("[]", "none", "0", "")
    if isinstance(actual, str):
        return cited.strip().lower() == actual.strip().lower()
    number = re.search(r"-?\d+(?:\.\d+)?", cited.replace(",", ""))
    return bool(number) and any(abs(x - actual) < 1e-6 for x in (float(number[0]), float(number[0]) / 100))


def check(case_id, cited):
    """Each cited fact, marked ok if the data confirms it ("a.b" paths count as "b")."""
    f = facts(case_id)
    data = {**records[case_id], **f["case"], **f["typical_for_this_care_type"],
            **f["compared_with_usual"], "shares_claim_number_with": f["shares_claim_number_with"],
            "hard_evidence": f["hard_evidence"]}
    return [{**c, "ok": (k := c["field"].split(".")[-1]) in data and matches(c["value"], data[k])}
            for c in cited]


def warnings(case_id):
    a = assessments[case_id]
    found = [f"The AI said {label(c['field']).lower()} is {c['value']}, which the data doesn't confirm"
             for c in check(case_id, a["key_facts"]) if not c["ok"]]
    if a["risk_level"] != ("High" if a["risk_score"] >= 50 else "Medium" if a["risk_score"] >= 20 else "Low"):
        found.append(f"The AI's score ({a['risk_score']}) doesn't match its risk level ({a['risk_level']})")
    if facts(case_id)["hard_evidence"] and (a["risk_level"] == "Low"
                                            or a["recommended_action"] == "Close as false positive"):
        found.append("The AI would close this case or rate it Low despite hard evidence")
    return found

## 4. Plain-English labels and the look of the app

In [ ]:
LABELS = {
    "case_id": "Case", "claim_number": "Claim number", "claim_date": "Claim date",
    "care_type": "Type of care", "claim_amount_usd": "Amount billed", "state": "State",
    "duplicate_service_billed": "Same service billed twice", "weekly_visit_frequency": "Visits per week",
    "member_provider_distance_miles": "Distance from patient to provider (miles)",
    "prior_claims_last_12mo": "Claims in the past year",
    "shared_contact_with_provider": "Patient and provider share contact details",
    "weekend_billing_ratio": "Bills on weekends", "round_dollar_billing_ratio": "Bills in round dollars",
    "amount_vs_peer_avg_pct": "Amount vs similar claims (%)",
    "referral_system_amount_vs_similar_pct": "Amount vs similar claims (%)",
    "recent_policy_change_flag": "Policy changed just before the claim",
    "service_overlap_other_provider": "Another provider billed for the same time",
    "shares_claim_number_with": "Same claim number also on", "hard_evidence": "Hard evidence",
}
FLAGS = {*HARD, "shared_contact_with_provider", "recent_policy_change_flag"}
SHORT = {"claim_amount_usd": "Amount", "weekly_visit_frequency": "Visits",
         "member_provider_distance_miles": "Distance"}


def label(field):
    field = field.split(".")[-1]
    base = field.removeprefix("typical_").removesuffix("_times_usual")
    if base != field:
        return f"{LABELS.get(base, base)}, {'typical' if field.startswith('typical_') else 'times the usual'}"
    return LABELS.get(field, field.replace("_", " ").capitalize())


def show(field, value):
    if field in FLAGS:
        return "Yes" if value else "No"
    if field.endswith("_ratio"):
        return f"{value:.0%} of bills"
    return f"${value:,.0f}" if field == "claim_amount_usd" else str(value)


def concerns(i):
    """What stands out, most serious first: hard evidence, yes/no red flags, big gaps from usual."""
    f, c, twins = facts(i), records[i], facts(i)["shares_claim_number_with"]
    return ([LABELS[k] for k in HARD if c[k] == 1] + [f"Same claim number as {', '.join(twins)}"] * bool(twins)
            + [LABELS[k] for k in ("shared_contact_with_provider", "recent_policy_change_flag") if c[k]]
            + [f"{SHORT[k[:-12]]} {v:g}× the usual" for k, v in f["compared_with_usual"].items() if v >= 2])


def badge(level, text=None):
    return f"<span class='badge {level.lower()}'>{text or level}</span>"


def ul(items, cls=""):
    return f"<ul class='{cls}'>" + "".join(f"<li>{x}</li>" for x in items) + "</ul>"


def cited(items):
    return ul((f"{'✓' if c['ok'] else '⚠'} <b>{label(c['field'])}:</b> {escape(c['value'])}"
               f"<br><span class='muted'>{escape(c['why'])}</span>" for c in items), "cites")


TIERS = {"High": ("#b42318", "#fde8e6"), "Medium": ("#b54708", "#fef0e1"), "Low": ("#067647", "#e3f6ec")}
CSS = "<style>@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');" + """
.cr-app { background: #f7f7f5; color: #1f2328; font: 14px/1.5 Inter, system-ui, sans-serif;
  border: 1px solid #e5e7eb; border-radius: 12px; overflow: hidden; }
.cr-app .muted { color: #6b7280; }  .cr-app ul { margin: 4px 0; padding-left: 18px; }
.cr-app ul.cites { list-style: none; padding-left: 0; }  .cites li { margin: 6px 0; }
.cr-top { background: #fff; border-bottom: 1px solid #e5e7eb; padding: 8px 20px; }
.brand { display: flex; gap: 10px; align-items: center; line-height: 1.2; }
.logo { width: 32px; height: 32px; border-radius: 8px; background: #2b4acb; color: #fff;
  display: grid; place-items: center; font-weight: 700; }
.head { padding: 12px 20px 0; }  .head h1 { font-size: 26px; margin: 2px 0; letter-spacing: -0.02em; }
.eyebrow { color: #2b4acb; font-size: 12px; font-weight: 600; text-transform: uppercase; letter-spacing: .06em; }
.cards { display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; padding: 12px 20px 4px; }
.card, .score { background: #fff; border: 1px solid #e5e7eb; border-top: 3px solid #2b4acb;
  border-radius: 10px; padding: 10px 14px; }
.label { font-size: 12px; font-weight: 600; color: #6b7280; text-transform: uppercase; letter-spacing: .04em; }
.n { font-size: 28px; font-weight: 700; line-height: 1.2; }  .n small { font-size: 15px; color: #6b7280; }
.bar { height: 6px; background: #e5e7eb; border-radius: 999px; overflow: hidden; margin: 6px 0; }
.bar span { display: block; height: 100%; background: #2b4acb; }
.badge { border-radius: 999px; padding: 1px 9px; font-size: 12px; font-weight: 600; white-space: nowrap; }
.alert { background: #fef0e1; color: #b54708; border-left: 3px solid #b54708; border-radius: 8px;
  padding: 8px 12px; margin: 8px 0; }
.cr-alerts { padding: 0 20px; }
.cr-tools { padding: 6px 20px 10px; align-items: center; gap: 12px; }
.cr-pills .widget-toggle-button { border-radius: 999px; background: #fff; border: 1px solid #e5e7eb;
  font-size: 13px; width: auto; padding: 0 12px; margin-right: 6px; }
.cr-pills .widget-toggle-button.mod-active { background: #2b4acb; color: #fff; }
.cr-primary { background: #2b4acb !important; color: #fff !important; border-radius: 6px; }
.cr-queue { background: #fff; border: 1px solid #e5e7eb; border-radius: 10px; margin: 0 0 20px 20px; }
.cr-row { border-bottom: 1px solid #e5e7eb; padding: 8px 10px; align-items: flex-start; }
.cr-row.selected { background: #eef1fd; box-shadow: inset 3px 0 0 #2b4acb; }
.cr-row.reviewed { opacity: .6; }  .cr-row.selected.reviewed { opacity: 1; }
.cr-row.head { background: #fafafa; color: #6b7280; font-size: 11.5px; font-weight: 600; text-transform: uppercase; }
.cells { display: grid; grid-template-columns: 110px 120px 1fr 1fr; gap: 12px; }
.clamp { display: -webkit-box; -webkit-line-clamp: 2; -webkit-box-orient: vertical; overflow: hidden; }
.cr-case { background: none !important; box-shadow: none !important; border: none !important; color: #2b4acb !important;
  font-weight: 700; padding: 0 !important; text-align: left; }
.warn, .over { color: #b54708; font-size: 13px; }  .done { color: #067647; }  .err { color: #b42318; }
.cr-side { background: #fff; border: 1px solid #e5e7eb; border-radius: 10px; margin: 0 20px 20px 12px; }
.cr-side h2 { margin: 0; font-size: 20px; }  .cr-side h3 { font-size: 14px; color: #6b7280; margin: 14px 0 4px; }
.cr-scroll { padding: 4px 16px 12px; }  .cr-nav { padding: 12px 16px 8px; border-bottom: 1px solid #e5e7eb; }
.cr-decision { background: #f7f7f5; border: 1px solid #e5e7eb; border-radius: 8px; padding: 10px; margin: 10px 0; }
.msg-q { background: #2b4acb; color: #fff; border-radius: 8px; padding: 6px 10px; margin: 6px 0 6px auto;
  width: fit-content; max-width: 85%; }
.msg-a { background: #f7f7f5; border-radius: 8px; padding: 6px 10px; }
details { margin-top: 12px; }  td { padding: 2px 8px 2px 0; vertical-align: top; }
""" + "".join(f".badge.{t.lower()} {{ background: {bg}; color: {fg}; }} .card.{t.lower()}, "
              f".score.{t.lower()} {{ border-top-color: {fg}; }} .card.{t.lower()} .n {{ color: {fg}; }} "
              f".score.{t.lower()} .bar span {{ background: {fg}; }}\n" for t, (fg, bg) in TIERS.items()) + "</style>"

## 5. Review the queue
Highest AI risk first. Click a case to open it: see the AI's assessment, accept or override its
call, leave notes and ask it questions. Decisions and notes are appended to `outputs/audit.csv`,
the audit trail (never edited).

In [ ]:
from collections import Counter

import ipywidgets as W
from IPython.display import display

AUDIT = Path("outputs/audit.csv")
COLUMNS = ["time", "case_id", "investigator", "kind", "risk_level", "action", "text",
           "ai_risk_level", "ai_action"]
HINTS = {"High": "Likely to need the fraud team", "Medium": "Need records or a closer look",
         "Low": "Likely false alarms"}
SUGGESTED = ["Why this risk level?", "How does this compare with similar cases?", "What should I request first?"]
queue = sorted(assessments, key=lambda i: -assessments[i]["risk_score"])
serial = {i: n for n, i in enumerate(queue, 1)}
WARN = {i: warnings(i) for i in queue}
look = {i: bool(WARN[i] or facts(i)["hard_evidence"]) for i in queue}  # needs a closer look
clean_low = {i: not look[i] and assessments[i]["risk_level"] == "Low"
             and assessments[i]["recommended_action"] == "Close as false positive" for i in queue}
state = {"case": None, "shown": queue, "latest": {}}  # latest: current decision per case
chats = {}  # case_id -> earlier questions and answers
counts = Counter(a["risk_level"] for a in assessments.values())
FILTERS = {"all": lambda i: True, "look": look.get,
           **{t: (lambda i, t=t: assessments[i]["risk_level"] == t) for t in TIERS}}


def trail():
    return pd.read_csv(AUDIT, dtype=str).fillna("") if AUDIT.exists() else pd.DataFrame(columns=COLUMNS)


def log(i, kind, **row):
    a = assessments[i]
    AUDIT.parent.mkdir(exist_ok=True)
    pd.DataFrame([{"time": datetime.now().isoformat(timespec="seconds"), "case_id": i,
                   "investigator": name.value.strip(), "kind": kind, "ai_risk_level": a["risk_level"],
                   "ai_action": a["recommended_action"], **row}], columns=COLUMNS).to_csv(
        AUDIT, mode="a", header=not AUDIT.exists(), index=False)


def say(message=""):
    status.value = f"<p class='err'>⚠ {message}</p>" if message else ""


# --- Header, summary cards and toolbar ---
today = datetime.now()
name = W.Text(placeholder="Your name", layout=W.Layout(width="200px"))
cards = W.HTML()
twins = sorted({tuple(sorted([i, *facts(i)["shares_claim_number_with"]])) for i in queue
                if facts(i)["shares_claim_number_with"]})
pills = W.ToggleButtons(options=[(f"All ({len(queue)})", "all"),
                                 (f"Needs a closer look ({sum(look.values())})", "look"),
                                 *[(f"{t} ({counts[t]})", t) for t in TIERS]]).add_class("cr-pills")
hide = W.Checkbox(description="Hide reviewed", indent=False, layout=W.Layout(width="140px"))
bulk = W.Button(layout=W.Layout(width="230px")).add_class("cr-primary")

# --- Queue: one row per case; the case ID opens it ---
rows = {}
for i in queue:
    rows[i] = W.HBox([W.HTML(f"<span class='muted'>{serial[i]}</span>", layout=W.Layout(width="32px")),
                      W.Button(description=i, layout=W.Layout(width="70px")).add_class("cr-case"),
                      W.HTML(layout=W.Layout(flex="1"))]).add_class("cr-row")
    rows[i].children[1].on_click(lambda _, i=i: open_case(i))
header = W.HBox([W.HTML("#", layout=W.Layout(width="32px")), W.HTML("Case", layout=W.Layout(width="70px")),
                 W.HTML("<div class='cells'><span>AI risk</span><span>Status</span><span>AI assessment</span>"
                        "<span>Main concerns</span></div>", layout=W.Layout(flex="1"))]).add_class("cr-row head")
queue_box = W.VBox(layout=W.Layout(flex="1", height="760px", overflow_y="auto")).add_class("cr-queue")

# --- Sidebar: the open case ---
title, position, status, case_html, decided, details, extra, thread = (W.HTML() for _ in range(8))
prev_btn, next_btn = (W.Button(description=d, layout=W.Layout(width="36px")) for d in "↑↓")
accept = W.Button(layout=W.Layout(width="auto")).add_class("cr-primary")
override = W.Button(description="Override…", layout=W.Layout(width="auto"))
new_level = W.Dropdown(options=list(TIERS)[::-1], description="Risk level")
new_action = W.Dropdown(options=list(Action.__args__), description="Action")
reason = W.Textarea(placeholder="Why are you deciding differently from the AI?", layout=W.Layout(width="95%"))
save, cancel = W.Button(description="Save override").add_class("cr-primary"), W.Button(description="Cancel")
choices = W.HBox([accept, override])
form = W.VBox([new_level, new_action, reason, W.HBox([save, cancel])], layout=W.Layout(display="none"))
note = W.Textarea(placeholder="Add a note for the file…", layout=W.Layout(width="95%"))
add_note = W.Button(description="Add note")
question = W.Text(placeholder="e.g. Why is the distance a concern?", layout=W.Layout(width="75%"))
ask_btn = W.Button(description="Ask", layout=W.Layout(width="60px")).add_class("cr-primary")
suggestions = [W.Button(description=q, layout=W.Layout(width="auto")) for q in SUGGESTED]
chat = W.Accordion(children=[W.VBox([thread, W.HBox(suggestions, layout=W.Layout(flex_flow="row wrap")),
                                     W.HBox([question, ask_btn])])], selected_index=0)
chat.set_title(0, "Ask about this case")


def render_row(i):
    a, d, con, w = assessments[i], state["latest"].get(i), concerns(i), WARN[i]
    status_ = ("<span class='muted'>To review</span>" if not d else "<span class='done'>✓ Accepted</span>"
               if d["kind"] == "accept" else f"<span class='over'>Overridden → {d['risk_level']}</span>")
    more = f" <span class='muted'>+{len(con) - 3} more</span>" if len(con) > 3 else ""
    check_note = f"<div class='warn'>⚠ {len(w)} thing{'s' * (len(w) > 1)} to double-check</div>" if w else ""
    rows[i].children[2].value = (
        f"<div class='cells'><span><b>{a['risk_score']}</b> {badge(a['risk_level'])}</span><span>{status_}</span>"
        f"<span><b>{a['recommended_action']}</b><div class='clamp'>{escape(a['summary'])}</div>{check_note}</span>"
        f"<span><div class='clamp'>{', '.join(con[:3]) or '<span class=muted>No warning signs</span>'}{more}</div>"
        "</span></div>")
    (rows[i].add_class if d else rows[i].remove_class)("reviewed")


def refresh():
    """Re-read the audit trail and redraw the cards, the queue and the bulk button."""
    t = trail()
    state["latest"] = t[t.kind != "note"].groupby("case_id").last().to_dict("index")
    done = len(state["latest"])
    cards.value = "<div class='cards'>" + "".join(
        f"<div class='card {t.lower()}'><div class='label'>{t} risk</div><div class='n'>{counts[t]}</div>"
        f"<div class='muted'>{HINTS[t]}</div></div>" for t in TIERS) + (
        f"<div class='card'><div class='label'>Reviewed</div><div class='n'>{done}<small> / {len(queue)}</small>"
        f"</div><div class='bar'><span style='width:{100 * done / len(queue)}%'></span></div></div></div>")
    for i in queue:
        render_row(i)
    pending = [i for i in queue if clean_low[i] and i not in state["latest"]]
    bulk.description, bulk.disabled = f"Accept {len(pending)} clean Low cases", not pending
    show_rows()


def show_rows(_=None):
    test = FILTERS[pills.value]
    state["shown"] = [i for i in queue if test(i) and not (hide.value and i in state["latest"])]
    queue_box.children = [header, *(rows[i] for i in state["shown"])] if state["shown"] else [
        header, W.HTML("<p class='muted' style='padding:20px'>No cases match this view.</p>")]
    if state["shown"] and state["case"] not in state["shown"]:
        open_case(state["shown"][0])  # keep the open case in step with the view
    else:
        place()


def place():
    k, shown = (state["shown"].index(state["case"]) if state["case"] in state["shown"] else -1), state["shown"]
    position.value = f"<span class='muted'>{k + 1} of {len(shown)}</span>" if k >= 0 else ""
    prev_btn.disabled, next_btn.disabled = k <= 0, k < 0 or k >= len(shown) - 1


def open_case(i):
    if state["case"]:
        rows[state["case"]].remove_class("selected")
    state["case"] = i
    rows[i].add_class("selected")
    a, c, hard = assessments[i], records[i], facts(i)["hard_evidence"]
    title.value = f"<h2>#{serial[i]} · {i}</h2>"
    case_html.value = (
        f"<p class='muted'>{c['care_type']} · ${c['claim_amount_usd']:,} · {c['state']}</p>"
        f"<div class='score {a['risk_level'].lower()}'><div class='label'>AI risk score</div>"
        f"<div class='n'>{a['risk_score']}<small> / 100</small></div><div class='bar'><span style='width:"
        f"{a['risk_score']}%'></span></div>{badge(a['risk_level'], a['risk_level'] + ' risk')} "
        f"<span class='muted'>{a['confidence']} confidence</span></div>"
        + (f"<div class='alert'>Can't be closed: {'; '.join(hard)}.</div>" if hard else "")
        + f"<p><b>Suggested: {a['recommended_action']}</b></p><p>{escape(a['summary'])}</p>"
        + "".join(f"<div class='warn'>⚠ {escape(w)}</div>" for w in WARN[i]))
    accept.description = f"Accept: {a['recommended_action']}"
    new_level.value, new_action.value, reason.value = a["risk_level"], a["recommended_action"], ""
    form.layout.display, choices.layout.display = "none", ""
    say()
    show_details(i)
    show_chat(i)
    place()


def show_details(i):
    a, d, t = assessments[i], state["latest"].get(i), trail()
    t = t[t.case_id == i]
    notes, history = t[t.kind == "note"], t[t.kind != "note"]
    decided.value = "" if not d else (
        f"<p><b>{'Accepted the AI' if d['kind'] == 'accept' else 'Overrode the AI'}: {d['action']} "
        f"({d['risk_level']} risk)</b><br><span class='muted'>{d['investigator']} · {d['time']}</span>"
        + (f"<br>Reason: {escape(d['text'])}" if d["text"] else "") + "</p>")
    details.value = (
        f"<h3>What the AI noticed</h3>{cited(check(i, a['key_facts']))}"
        f"<h3>What to do next</h3>{ul(map(escape, a['next_steps']))}"
        f"<h3>What this data can't tell us</h3>{ul(map(escape, a['open_questions']))}<h3>Notes</h3>"
        + (ul(f"{escape(r.text)}<br><span class='muted'>{r.investigator} · {r.time}</span>"
              for r in notes.itertuples()) if len(notes) else "<p class='muted'>No notes yet.</p>"))
    extra.value = (
        f"<details><summary>Decision history ({len(history)})</summary>"
        + (ul(f"{r.kind.capitalize()}: {r.action} ({r.risk_level} risk)"
              + (f" · Reason: {escape(r.text)}" if r.text else "")
              + f"<br><span class='muted'>{r.investigator} · {r.time} · AI had said {r.ai_action}</span>"
              for r in history.iloc[::-1].itertuples()) if len(history) else "<p class='muted'>No decisions yet.</p>")
        + "</details><details><summary>All claim details</summary><table>"
        + "".join(f"<tr><td class='muted'>{label(k)}</td><td>{escape(show(k, v))}</td></tr>" for k, v in records[i].items())
        + "</table></details>")


def show_chat(i):
    thread.value = "".join(
        f"<div class='msg-q'>{escape(t['question'])}</div><div class='msg-a'>{escape(t['answer'])}"
        + ("" if t["answerable"] else "<div class='warn'>The case data can't fully answer this.</div>")
        + cited(t["citations"]) + "</div>" for t in chats.get(i, []))


def decide(kind):
    i, a = state["case"], assessments[state["case"]]
    if not name.value.strip():
        return say("Enter your name at the top first.")
    if kind == "override":
        if not reason.value.strip():
            return say("Add a reason for the override.")
        if (new_level.value, new_action.value) == (a["risk_level"], a["recommended_action"]):
            return say("That matches the AI's call: use Accept, or change the risk level or action.")
        log(i, "override", risk_level=new_level.value, action=new_action.value, text=reason.value.strip())
    else:
        log(i, "accept", risk_level=a["risk_level"], action=a["recommended_action"])
    k = state["shown"].index(i) if i in state["shown"] else -1
    after = state["shown"][k + 1:] + state["shown"][:max(k, 0)]
    refresh()
    # Move on to the next case still to review, as an investigator would.
    open_case(next((j for j in after if j not in state["latest"] and j in state["shown"]), state["case"]))


def note_it(_):
    if not (name.value.strip() and note.value.strip()):
        return say("Enter your name at the top and write a note.")
    log(state["case"], "note", text=note.value.strip())
    note.value = ""
    show_details(state["case"])


def accept_clean_lows(_):
    if not name.value.strip():
        return say("Enter your name at the top first.")
    for i in queue:
        if clean_low[i] and i not in state["latest"]:
            log(i, "accept", risk_level="Low", action="Close as false positive")
    refresh()
    open_case(state["case"])


def ask(text):
    i, q, a = state["case"], text.strip(), assessments[state["case"]]
    if not q:
        return
    question.value = ""
    thread.value += f"<div class='msg-q'>{escape(q)}</div><div class='msg-a muted'>Thinking…</div>"
    earlier = [{"question": t["question"], "answer": t["answer"]} for t in chats.get(i, [])[-6:]]
    gist = {k: a[k] for k in ("summary", "risk_score", "risk_level", "recommended_action")}
    try:
        reply = ask_ai(QA, f"Case facts:\n{json.dumps(facts(i))}\nYour earlier assessment:\n{json.dumps(gist)}"
                           f"\nEarlier questions:\n{json.dumps(earlier)}\nQuestion: {q}", Answer)
    except RuntimeError as e:
        show_chat(i)
        thread.value += f"<p class='err'>⚠ {escape(str(e))}</p>"
        return
    chats.setdefault(i, []).append({**reply.model_dump(), "question": q,
                                    "citations": check(i, [c.model_dump() for c in reply.citations])})
    if state["case"] == i:
        show_chat(i)


def step(offset):
    k = state["shown"].index(state["case"]) + offset
    if 0 <= k < len(state["shown"]):
        open_case(state["shown"][k])


def show_form(visible):
    form.layout.display, choices.layout.display = ("", "none") if visible else ("none", "")
    say()


pills.observe(show_rows, "value")
hide.observe(show_rows, "value")
bulk.on_click(accept_clean_lows)
prev_btn.on_click(lambda _: step(-1))
next_btn.on_click(lambda _: step(1))
accept.on_click(lambda _: decide("accept"))
override.on_click(lambda _: show_form(True))
cancel.on_click(lambda _: show_form(False))
save.on_click(lambda _: decide("override"))
add_note.on_click(note_it)
ask_btn.on_click(lambda _: ask(question.value))
for b in suggestions:
    b.on_click(lambda b: ask(b.description))

refresh()
display(W.VBox([
    W.HTML(CSS),
    W.HBox([W.HTML("<div class='brand'><div class='logo'>✓</div><div><b>Case Review</b><br>"
                   "<span class='muted'>Junior AI Investigator</span></div></div>"),
            W.HBox([W.HTML("<span class='muted'>Reviewing as&nbsp;</span>"), name])],
           layout=W.Layout(justify_content="space-between", align_items="center")).add_class("cr-top"),
    W.HTML(f"<div class='head'><div class='eyebrow'>{today:%A, %B} {today.day}</div><h1>Today's fraud "
           f"referrals</h1><div class='muted'>{len(queue)} cases flagged by the fraud engine, highest AI risk "
           "first. The AI scores and explains each case; every fact it cites is checked.</div></div>"),
    cards,
    W.HTML("<div class='cr-alerts'>" + "".join(f"<div class='alert'>Same claim number on {' and '.join(g)}</div>"
                                               for g in twins) + "</div>"),
    W.HBox([pills, hide, bulk], layout=W.Layout(flex_flow="row wrap")).add_class("cr-tools"),
    W.HBox([queue_box, W.VBox([
        W.HBox([title, W.HBox([position, prev_btn, next_btn])],
               layout=W.Layout(justify_content="space-between", align_items="center")).add_class("cr-nav"),
        W.VBox([case_html, W.VBox([W.HTML("<b>Your decision</b>"), decided, choices, form, status]).add_class(
            "cr-decision"), details, note, add_note, extra], layout=W.Layout(height="520px", overflow_y="auto")
        ).add_class("cr-scroll"),
        chat], layout=W.Layout(width="460px", height="760px")).add_class("cr-side")]),
]).add_class("cr-app"))